# 06 — Modern Story Building

Full pipeline implementation using modern approaches as alternatives to the traditional NewsLens paper method.

Two approaches implemented:
- **Option A**: Same sliding window + Louvain + majority vote structure as the paper, but using BGE-M3 embedding similarity instead of keyword Jaccard for graph edges
- **Option B**: BERTopic — direct clustering on BGE-M3 embeddings, no sliding windows or keyword graphs

In [3]:
from pathlib import Path
import pickle
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity as cos_sim_sk
from community import community_louvain
from collections import Counter
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize

REPO_ROOT = Path("../").resolve()
PARQUET_PATH = REPO_ROOT / "data" / "processed" / "articles_clean.parquet"
RANDOM_SEED = 42

# Load data

In [4]:
df = pd.read_parquet(PARQUET_PATH)
df["published_at_dt"] = pd.to_datetime(df["published_at_dt"], utc=True)

dates = (
    df["published_at_dt"].dt.tz_localize(None)
    if df["published_at_dt"].dt.tz is not None
    else df["published_at_dt"]
)
dates = dates.dt.normalize()

print(f"Articles: {len(df)}")

Articles: 30462


In [8]:
embs_all = np.load(REPO_ROOT / "data" / "processed" / "embeddings.npy")
emb_index_all = {i: embs_all[i] for i in range(len(df))}

print(f"Embeddings: {embs_all.shape}")

Embeddings: (30462, 1024)


# Sliding windows

In [4]:
N_DAYS = 5
STEP = 2

unique_days = sorted(dates.unique())
windows = [
    (unique_days[i], unique_days[min(i + N_DAYS - 1, len(unique_days) - 1)])
    for i in range(0, len(unique_days) - N_DAYS + 1, STEP)
]
print(f"Windows: {len(windows)}")

Windows: 15


# Evaluation function

In [5]:
def evaluate_stories(
    stories,
    emb_index_all,
    total_articles,
    n_samples=10000,
    random_seed=42,
    verbose=True,
):
    rng = np.random.default_rng(random_seed)

    cohesions, sizes = [], []
    for sid, arts in stories.items():
        if len(arts) < 3:
            continue
        embs = np.array([emb_index_all[i] for i in arts])
        sim = cos_sim_sk(embs)
        n = len(arts)
        pairs = [sim[i, j] for i in range(n) for j in range(i + 1, n)]
        cohesions.append(np.mean(pairs))
        sizes.append(n)

    story_ids = [sid for sid, arts in stories.items() if len(arts) >= 3]
    inter_scores = []
    while len(inter_scores) < n_samples:
        sid_a, sid_b = rng.choice(story_ids, 2, replace=False)
        a = int(rng.choice(stories[sid_a]))
        b = int(rng.choice(stories[sid_b]))
        inter_scores.append(
            float(cos_sim_sk([emb_index_all[a]], [emb_index_all[b]])[0][0])
        )

    in_stories = sum(len(arts) for arts in stories.values() if len(arts) >= 3)
    coverage = in_stories / total_articles

    mean_coh = np.average(cohesions, weights=sizes)
    mean_sep = np.mean(inter_scores)
    ratio = mean_coh / mean_sep if mean_sep > 0 else float("inf")

    if verbose:
        print(f"Stories evaluated:     {len(cohesions)}")
        print(f"Coverage:              {coverage:.1%} ({in_stories}/{total_articles})")
        print(f"Intra-story coherence: {mean_coh:.4f}")
        print(f"Inter-story similarity:{mean_sep:.4f}")
        print(f"Ratio (coh/sim):       {ratio:.2f}x")

    return {
        "coherence": mean_coh,
        "separation": mean_sep,
        "ratio": ratio,
        "coverage": coverage,
    }

# Option A — Embedding graph + Louvain + majority vote

Keeps the same sliding window structure from the paper.
Replaces keyword Jaccard with BGE-M3 cosine similarity for graph edges.
Story building via majority vote + cross-gap cosine merge stays the same.

In [8]:
def build_topics_embed(
    df,
    emb_index_all,
    dates,
    windows,
    min_similarity: float = 0.5,
    random_seed: int = RANDOM_SEED,
):
    topics = []

    for ws, we in tqdm(windows, desc="Building embedding topics"):
        mask = (dates >= ws) & (dates <= we)
        window_indices = list(df[mask].index)

        if len(window_indices) < 2:
            topics.append({})
            continue

        embs = np.array([emb_index_all[i] for i in window_indices])
        sim_matrix = cos_sim_sk(embs)

        G = nx.Graph()
        G.add_nodes_from(range(len(window_indices)))

        for i in range(len(window_indices)):
            for j in range(i + 1, len(window_indices)):
                if sim_matrix[i, j] >= min_similarity:
                    G.add_edge(i, j, weight=float(sim_matrix[i, j]))

        partition = community_louvain.best_partition(G, random_state=random_seed)

        topic_map = {}
        for local_idx, topic_id in partition.items():
            topic_map[window_indices[local_idx]] = topic_id

        topics.append(topic_map)

    return topics

In [12]:
def build_stories_embed(window_topics, emb_index_all, T3=0.8):
    article_story = {}
    story_alias = {}
    story_last_win = {}
    next_id = [0]

    def resolve(sid):
        while story_alias.get(sid, sid) != sid:
            sid = story_alias[sid]
        return sid

    def create_story(arts, win_idx):
        sid = next_id[0]
        next_id[0] += 1
        for a in arts:
            article_story[a] = sid
        story_last_win[sid] = win_idx

    def assign_to_story(arts, sid, win_idx):
        for a in arts:
            article_story[a] = sid
        story_last_win[sid] = win_idx

    def merge_stories(sid_keep, sid_drop):
        sid_keep = resolve(sid_keep)
        sid_drop = resolve(sid_drop)
        if sid_keep == sid_drop:
            return
        story_alias[sid_drop] = sid_keep

    for win_idx, partition in enumerate(tqdm(window_topics, desc="Linking")):
        clusters = defaultdict(list)
        for art_idx, cluster_id in partition.items():
            clusters[cluster_id].append(art_idx)
        for arts in clusters.values():
            prev = [resolve(article_story[a]) for a in arts if a in article_story]
            if not prev:
                create_story(arts, win_idx)
                continue
            vote = Counter(prev)
            top_sid, top_count = vote.most_common(1)[0]
            if top_count > len(arts) / 2:
                for other in list(vote.keys()):
                    if other != top_sid:
                        merge_stories(top_sid, other)
                assign_to_story(arts, top_sid, win_idx)
            else:
                create_story(arts, win_idx)

    # cross-gap merge using mean embeddings instead of keyword vectors
    story_articles_tmp = defaultdict(list)
    for art_idx, sid in article_story.items():
        story_articles_tmp[resolve(sid)].append(art_idx)

    canonical_ids = sorted(story_articles_tmp.keys())
    mean_embs = np.array(
        [
            np.mean([emb_index_all[a] for a in story_articles_tmp[sid]], axis=0)
            for sid in canonical_ids
        ]
    )
    mean_embs = mean_embs / np.linalg.norm(mean_embs, axis=1, keepdims=True)
    sim_matrix = mean_embs @ mean_embs.T

    merged_count = 0
    for ri in range(len(canonical_ids)):
        for ci in range(ri + 1, len(canonical_ids)):
            if sim_matrix[ri, ci] >= T3:
                sid_a = resolve(canonical_ids[ri])
                sid_b = resolve(canonical_ids[ci])
                if sid_a == sid_b:
                    continue
                if (
                    abs(story_last_win.get(sid_a, -1) - story_last_win.get(sid_b, -1))
                    > 1
                ):
                    merge_stories(sid_a, sid_b)
                    merged_count += 1

    story_articles = defaultdict(list)
    for art_idx, sid in article_story.items():
        story_articles[resolve(sid)].append(art_idx)
    story_articles = dict(story_articles)

    sizes = pd.Series({sid: len(arts) for sid, arts in story_articles.items()})
    print(
        f"T3={T3} | Stories: {len(story_articles):,} | Cross-gap merges: {merged_count}"
    )
    print(
        f"Singletons: {(sizes==1).sum()} ({(sizes==1).mean():.1%}) | Largest: {sizes.max()}"
    )
    return story_articles, article_story, resolve

In [14]:
topics_a = build_topics_embed(df, emb_index_all, dates, windows, min_similarity=0.5)
stories_a, _, _ = build_stories_embed(topics_a, emb_index_all, T3=0.8)

print("\n=== Option A ===")
r_a = evaluate_stories(stories_a, emb_index_all, len(df))

Building embedding topics:   0%|          | 0/15 [00:00<?, ?it/s]

Linking:   0%|          | 0/15 [00:00<?, ?it/s]

T3=0.8 | Stories: 613 | Cross-gap merges: 0
Singletons: 503 (82.1%) | Largest: 29599

=== Option A ===
Stories evaluated:     38
Coverage:              97.9% (29815/30462)
Intra-story coherence: 0.3161
Inter-story similarity:0.2853
Ratio (coh/sim):       1.11x


min_similarity=0.5 is too low for Bulgarian news embeddings. Within a 5-day window, most articles score above 0.5 with each other because they all share Bulgarian news context (politics, economy, common entities). Louvain sees a nearly fully-connected graph and merges everything into one massive community.

This is a fundamental difference from keyword Jaccard — Jaccard naturally goes to 0 for unrelated articles (no shared keywords). Embeddings never go to 0 for Bulgarian news; there's always a semantic floor of ~0.3.

In [15]:
topics_a = build_topics_embed(df, emb_index_all, dates, windows, min_similarity=0.7)
stories_a, _, _ = build_stories_embed(topics_a, emb_index_all, T3=0.8)

print("\n=== Option A (min_similarity=0.7) ===")
r_a = evaluate_stories(stories_a, emb_index_all, len(df))

Building embedding topics:   0%|          | 0/15 [00:00<?, ?it/s]

Linking:   0%|          | 0/15 [00:00<?, ?it/s]

T3=0.8 | Stories: 13,781 | Cross-gap merges: 761
Singletons: 10330 (75.0%) | Largest: 1124

=== Option A (min_similarity=0.7) ===
Stories evaluated:     1675
Coverage:              54.4% (16580/30462)
Intra-story coherence: 0.6589
Inter-story similarity:0.3071
Ratio (coh/sim):       2.15x


In [16]:
story_sizes_a = sorted(stories_a.items(), key=lambda x: -len(x[1]))
sid, arts = story_sizes_a[0]

print(f"Largest story: {len(arts)} articles")
print(
    f"Date range: {df.iloc[arts[0]]['published_at_dt'].strftime('%d %b')} → {df.iloc[arts[-1]]['published_at_dt'].strftime('%d %b')}"
)
print("\nSample of 20 titles:")
for idx in sorted(arts, key=lambda i: df.iloc[i]["published_at_dt"])[:20]:
    pub = df.iloc[idx]["published_at_dt"].strftime("%d %b %H:%M")
    print(f"  {pub} - {df.iloc[idx]['source']} - {df.iloc[idx]['title']}")

Largest story: 1124 articles
Date range: 23 Apr → 21 May

Sample of 20 titles:
  20 Apr 12:27 - 24chasa - ЦИК при 100% преброени: "Прогресивна България" печели изборите с 44.594%, ГЕРБ са втори
  20 Apr 12:28 - 24chasa - Коалицията на Румен Радев първа и в област Монтана с 47,27%
  20 Apr 12:31 - 24chasa - Окончателно в Благоевград: Радев с 39,84%, ГЕРБ - 19,76%, ДПС с 13%
  20 Apr 12:42 - actualno - ЦИК пусна финалните резултати от изборите. Кой колко депутати ще има? (ГРАФИКИ)
  20 Apr 13:42 - 24chasa - Вижте как се разпределя вотът по избирателни райони
  20 Apr 15:02 - 24chasa - Илияна Йотова: Следващата седмица най-вероятно ще свикам 52-рото Народно събрание
  20 Apr 15:26 - actualno - Кога ще бъде свикано новото народно събрание: Президентът Илияна Йотова с първи коментар след вота
  20 Apr 18:04 - 24chasa - Радев би и по преференции – 3 пъти повече от Борисов в София и 7 пъти от Бойко Рашков
  20 Apr 19:00 - 24chasa - Радев привлече 28% от ГЕРБ, 36% от ПП-ДБ и 52% от БСП (График

## Largest Story Inspection — Option A (min_similarity=0.7)

The largest story contains **1,124 articles spanning April 20 → May 21** (one month).

Inspecting the titles reveals it captured all Bulgarian election coverage and post-election
politics — election results, parliamentary formation, government negotiations, budget
discussions — merged into one cluster because they all share the same political
context (Радев, Йотова, ЦИК, НС, ГЕРБ).

This is the core limitation of embedding-based similarity for Bulgarian political news:
shared political entities and vocabulary create a semantic "floor" that connects
otherwise distinct events within the same broad political narrative.


In [19]:
for sim in [0.72, 0.75, 0.78]:
    topics_test = build_topics_embed(
        df, emb_index_all, dates, windows, min_similarity=sim
    )
    stories_test, _, _ = build_stories_embed(topics_test, emb_index_all, T3=0.8)
    r = evaluate_stories(stories_test, emb_index_all, len(df), verbose=False)
    sizes = pd.Series({sid: len(arts) for sid, arts in stories_test.items()})
    print(
        print(
            f"sim={sim} | stories={len(stories_test):,} | largest={sizes.max()} | coherence={r['coherence']:.4f} | separation={r['separation']:.4f} | coverage={r['coverage']:.1%} | ratio={r['ratio']:.2f}"
        )
    )

Building embedding topics:   0%|          | 0/15 [00:00<?, ?it/s]

Linking:   0%|          | 0/15 [00:00<?, ?it/s]

T3=0.8 | Stories: 15,011 | Cross-gap merges: 824
Singletons: 11405 (76.0%) | Largest: 829
sim=0.72 | stories=15,011 | largest=829 | coherence=0.6930 | separation=0.3088 | coverage=50.5% | ratio=2.24
None


Building embedding topics:   0%|          | 0/15 [00:00<?, ?it/s]

Linking:   0%|          | 0/15 [00:00<?, ?it/s]

T3=0.8 | Stories: 16,774 | Cross-gap merges: 945
Singletons: 13014 (77.6%) | Largest: 634
sim=0.75 | stories=16,774 | largest=634 | coherence=0.7386 | separation=0.3105 | coverage=44.4% | ratio=2.38
None


Building embedding topics:   0%|          | 0/15 [00:00<?, ?it/s]

Linking:   0%|          | 0/15 [00:00<?, ?it/s]

T3=0.8 | Stories: 18,473 | Cross-gap merges: 1091
Singletons: 14635 (79.2%) | Largest: 236
sim=0.78 | stories=18,473 | largest=236 | coherence=0.7791 | separation=0.3109 | coverage=38.5% | ratio=2.51
None


## Option A — Parameter Sweep Results

Tested `min_similarity` values from 0.50 to 0.78, keeping T3=0.8 fixed.
Traditional baseline included for comparison (T2=6, T3=0.8).

| Method | min_similarity | Coherence | Separation | Coverage | Ratio | Largest story |
|---|---|---|---|---|---|---|
| Traditional (keyword Jaccard) | T2=6 | 0.649 | 0.313 | 36.0% | 2.07x | 497 art |
| Option A (embedding graph) | 0.50 | 0.316 | 0.285 | 97.9% | 1.11x | 29,599 art |
| Option A (embedding graph) | 0.70 | 0.659 | 0.307 | 54.4% | 2.15x | 1,124 art |
| Option A (embedding graph) | 0.72 | 0.693 | 0.309 | 50.5% | 2.24x | 829 art |
| Option A (embedding graph) | 0.75 | 0.739 | 0.311 | 44.4% | 2.38x | 634 art |
| **Option A (embedding graph)** | **0.78** | **0.779** | **0.311** | **38.5%** | **2.51x** | **236 art** |

## Selected Parameters: min_similarity=0.78, T3=0.8

`min_similarity=0.78` is selected as the final Option A configuration because:

1. **Fair comparison with traditional** — coverage of 38.5% is close to the traditional
baseline (36%), making the comparison directly valid
2. **Significantly better coherence** — 0.779 vs 0.649 (+13%)
3. **Better ratio** — 2.51x vs 2.07x
4. **Smaller largest story** — 236 articles vs 497 in traditional
5. **Stable separation** — inter-story similarity remains flat at ~0.311 across all
configurations, confirming it is driven by the natural Bulgarian news semantic floor
rather than by parameter choice

`min_similarity=0.50` illustrates the lower bound — the natural semantic floor of
Bulgarian news embeddings (~0.3–0.5) means almost every article connects to every
other within a window, collapsing the entire dataset into a few giant clusters.

`min_similarity=0.75` is also a strong option if higher coverage (44.4%) is prioritised
over maximum coherence — it still outperforms traditional on all metrics while
capturing 8% more articles.

## Key Finding

Replacing keyword Jaccard with BGE-M3 semantic similarity **within the same
sliding window + Louvain + majority vote structure** consistently improves
story quality across all metrics at comparable coverage levels.
The improvement in coherence (+13%) confirms that semantic embeddings capture
topic boundaries more accurately than keyword co-occurrence for Bulgarian news.
The threshold must be set high (≥0.75) to overcome the natural semantic floor
of Bulgarian news embeddings caused by shared political context and vocabulary.
Inter-story similarity remains stable at ~0.31 regardless of parameters,
confirming it reflects shared Bulgarian news context rather than clustering quality.


# Option B — BERTopic

Direct clustering on BGE-M3 embeddings.
No sliding windows, no keyword graphs.
Topics discovered via HDBSCAN on the embedding space.

In [6]:
from bertopic import BERTopic
from umap import UMAP
from sklearn.cluster import HDBSCAN


def build_stories_bertopic(
    df,
    embs_all,
    dates,
    min_cluster_size: int = 5,
    max_days_gap: int = 7,
    random_seed: int = RANDOM_SEED,
):
    # Step 1: BERTopic on all articles
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=random_seed,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric="euclidean",
        cluster_selection_method="eom",
    )
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        language="multilingual",
        calculate_probabilities=False,
        verbose=True,
    )

    texts = [
        f"{df.iloc[i]['title'] or ''}. {(df.iloc[i]['full_text'] or '')[:150]}".strip()
        for i in range(len(df))
    ]
    topic_labels, _ = topic_model.fit_transform(texts, embeddings=embs_all)

    # Step 2: split each topic by time gap → stories
    stories = {}
    story_id = 0

    topic_to_articles = defaultdict(list)
    for art_idx, label in enumerate(topic_labels):
        if label == -1:  # noise → skip
            continue
        topic_to_articles[label].append(art_idx)

    for label, arts in topic_to_articles.items():
        sorted_arts = sorted(arts, key=lambda i: dates.iloc[i])
        current_story = [sorted_arts[0]]
        for i in range(1, len(sorted_arts)):
            gap = (dates.iloc[sorted_arts[i]] - dates.iloc[sorted_arts[i - 1]]).days
            if gap <= max_days_gap:
                current_story.append(sorted_arts[i])
            else:
                stories[story_id] = current_story
                story_id += 1
                current_story = [sorted_arts[i]]
        stories[story_id] = current_story
        story_id += 1

    sizes = pd.Series({sid: len(arts) for sid, arts in stories.items()})
    print(f"Stories: {len(stories):,}")
    print(
        f"Singletons: {(sizes==1).sum()} ({(sizes==1).mean():.1%}) | Largest: {sizes.max()}"
    )
    return stories, topic_labels

In [9]:
stories_b, topic_labels = build_stories_bertopic(df, embs_all, dates)

print("\n=== Option B: BERTopic ===")
r_b = evaluate_stories(stories_b, emb_index_all, len(df))

2026-05-30 23:43:22,618 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-30 23:43:41,730 - BERTopic - Dimensionality - Completed ✓
2026-05-30 23:43:41,731 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-30 23:43:43,683 - BERTopic - Cluster - Completed ✓
2026-05-30 23:43:43,689 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-30 23:43:44,262 - BERTopic - Representation - Completed ✓


Stories: 2,087
Singletons: 330 (15.8%) | Largest: 330

=== Option B: BERTopic ===
Stories evaluated:     1574
Coverage:              71.8% (21876/30462)
Intra-story coherence: 0.6181
Inter-story similarity:0.3088
Ratio (coh/sim):       2.00x


# Option B — BERTopic

## Approach

Replaces the entire traditional 3-step pipeline with a modern two-step approach:

1. **BERTopic** — clusters all 30,462 articles simultaneously using BGE-M3 embeddings.
   Internally uses UMAP for dimensionality reduction and HDBSCAN for clustering.
   No sliding windows, no keyword graphs, no Louvain.

2. **Time splitting** — within each BERTopic cluster, articles separated by more than
   7 days are split into separate stories. This separates distinct events that share
   the same broad topic (e.g. Trump tariffs vs Trump shooting — same topic, different stories).

## Parameters
- `min_cluster_size=5` — minimum articles per BERTopic cluster
- `max_days_gap=7` — time gap threshold for splitting clusters into stories
- Articles assigned label `-1` by HDBSCAN (noise) are excluded

## Results

| Metric | Value |
|---|---|
| Stories | 2,087 |
| Stories evaluated (≥3 articles) | 1,574 |
| Intra-story coherence | 0.6181 |
| Inter-story similarity | 0.3088 |
| Ratio (coh/sim) | 2.00x |
| Coverage | **71.8%** (21,876 / 30,462) |
| Singleton rate | **15.8%** (330 / 2,087) |
| Largest story | 330 articles |

## Key Observations

**Highest coverage (71.8%)** — BERTopic captures nearly twice as many articles as the
traditional approach (36%) and Option A (38.5%). HDBSCAN's density-based clustering
assigns most articles to a cluster rather than leaving them as orphans.

**Lowest singleton rate (15.8%)** — compared to ~75-79% for keyword/embedding graph
approaches. BERTopic naturally groups isolated articles with their nearest semantic
neighbours rather than leaving them unassigned.

**Lower coherence (0.618)** — the broader clusters include borderline articles that
the stricter graph-based approaches would leave ungrouped, slightly reducing
intra-story semantic similarity.

**Fewer but larger stories (2,087)** — compared to ~18,000+ for graph-based approaches.
BERTopic finds coarser semantic clusters which the time splitting then subdivides.

## Trade-off

BERTopic optimises for **recall** — it captures more of the news at the cost of
slightly lower precision. The graph-based approaches (traditional and Option A)
optimise for **precision** — tighter stories but more ungrouped articles.

This makes BERTopic the better choice when **completeness of coverage** is the
priority, and graph-based approaches better when **story purity** is paramount.


In [11]:
# get noise articles (label == -1)
noise_indices = [i for i, label in enumerate(topic_labels) if label == -1]
print(f"Noise articles: {len(noise_indices)} ({len(noise_indices)/len(df):.1%})")

print("\nSample of 20 excluded articles:")
import random

sample = random.sample(noise_indices, min(20, len(noise_indices)))
for idx in sorted(sample, key=lambda i: df.iloc[i]["published_at_dt"]):
    pub = df.iloc[idx]["published_at_dt"].strftime("%d %b %H:%M")
    print(
        f"  {pub} - {df.iloc[idx]['source']} - {df.iloc[idx]['title']} - {df.iloc[idx]['url']}"
    )

Noise articles: 7890 (25.9%)

Sample of 20 excluded articles:
  23 Apr 13:24 - fakti - Арда Гюлер аут до края на сезона - https://fakti.bg/sport/1050110-arda-guler-aut-do-kraa-na-sezona
  25 Apr 08:49 - nova - Млада жена загина при катастрофа с тротинетка в Пловдив - https://nova.bg/news/view/2026/04/25/535074/%D0%BC%D0%BB%D0%B0%D0%B4%D0%B0-%D0%B6%D0%B5%D0%BD%D0%B0-%D0%B7%D0%B0%D0%B3%D0%B8%D0%BD%D0%B0-%D0%BF%D1%80%D0%B8-%D0%BA%D0%B0%D1%82%D0%B0%D1%81%D1%82%D1%80%D0%BE%D1%84%D0%B0-%D1%81-%D1%82%D1%80%D0%BE%D1%82%D0%B8%D0%BD%D0%B5%D1%82%D0%BA%D0%B0-%D0%B2-%D0%BF%D0%BB%D0%BE%D0%B2%D0%B4%D0%B8%D0%B2/
  26 Apr 06:12 - 24chasa - Какво прави българина най-щастлив? Да има дете, обич и пътувания (Вижте резултатите от анкетата за щастието) - https://www.24chasa.bg/kolektsionerskiyat_broy/article/22729316
  26 Apr 10:01 - 24chasa - Макрон: Изразявам на Доналд Тръмп цялата си подкрепа - https://www.24chasa.bg/mezhdunarodni/article/22736238
  26 Apr 14:47 - 24chasa - Израел назначи посланик в Сомал

In [12]:
def build_stories_bertopic_without_noise_removal(
    df,
    embs_all,
    dates,
    min_cluster_size: int = 5,
    max_days_gap: int = 7,
    random_seed: int = RANDOM_SEED,
):
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=random_seed,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric="euclidean",
        cluster_selection_method="eom",
    )
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        language="multilingual",
        calculate_probabilities=False,
        verbose=True,
    )

    texts = [
        f"{df.iloc[i]['title'] or ''}. {(df.iloc[i]['full_text'] or '')[:150]}".strip()
        for i in range(len(df))
    ]
    topic_labels, _ = topic_model.fit_transform(texts, embeddings=embs_all)

    # assign noise articles to nearest topic
    new_topics = topic_model.reduce_outliers(
        texts, topic_labels, strategy="embeddings", embeddings=embs_all
    )
    n_still_noise = sum(1 for t in new_topics if t == -1)
    print(
        f"Noise before: {sum(1 for t in topic_labels if t==-1)} → after reduce_outliers: {n_still_noise}"
    )

    # split each topic by time gap → stories
    stories = {}
    story_id = 0

    topic_to_articles = defaultdict(list)
    for art_idx, label in enumerate(new_topics):
        if label == -1:
            # still unassigned → singleton story
            stories[story_id] = [art_idx]
            story_id += 1
        else:
            topic_to_articles[label].append(art_idx)

    for label, arts in topic_to_articles.items():
        sorted_arts = sorted(arts, key=lambda i: dates.iloc[i])
        current_story = [sorted_arts[0]]
        for i in range(1, len(sorted_arts)):
            gap = (dates.iloc[sorted_arts[i]] - dates.iloc[sorted_arts[i - 1]]).days
            if gap <= max_days_gap:
                current_story.append(sorted_arts[i])
            else:
                stories[story_id] = current_story
                story_id += 1
                current_story = [sorted_arts[i]]
        stories[story_id] = current_story
        story_id += 1

    sizes = pd.Series({sid: len(arts) for sid, arts in stories.items()})
    print(f"Stories: {len(stories):,}")
    print(
        f"Singletons: {(sizes==1).sum()} ({(sizes==1).mean():.1%}) | Largest: {sizes.max()}"
    )
    return stories, new_topics

In [14]:
stories_b_without_noise, topic_labels_without_noise = build_stories_bertopic_without_noise_removal(df, embs_all, dates)

print("\n=== BERTopic without Noise Removal ===")
r_b_w_n = evaluate_stories(stories_b_without_noise, emb_index_all, len(df))

2026-05-30 23:51:10,494 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-30 23:51:23,091 - BERTopic - Dimensionality - Completed ✓
2026-05-30 23:51:23,092 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-30 23:51:25,037 - BERTopic - Cluster - Completed ✓
2026-05-30 23:51:25,042 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-30 23:51:25,612 - BERTopic - Representation - Completed ✓


Noise before: 7890 → after reduce_outliers: 0
Stories: 2,006
Singletons: 260 (13.0%) | Largest: 399

=== BERTopic without Noise Removal ===
Stories evaluated:     1602
Coverage:              98.2% (29914/30462)
Intra-story coherence: 0.5810
Inter-story similarity:0.3091
Ratio (coh/sim):       1.88x


## Option B — BERTopic: Outlier Reduction

### Without outlier reduction
Articles labeled -1 (noise) by HDBSCAN are excluded entirely.
7,890 articles (~28%) are left unassigned.

### With outlier reduction (`reduce_outliers`)
Noise articles are reassigned to their nearest topic using embedding similarity.
All 7,890 noise articles successfully assigned → 0 remaining noise.

### Comparison

| | Without reduction | With reduction |
|---|---|---|
| Coverage | 71.8% | **98.2%** |
| Coherence | 0.618 | 0.581 |
| Ratio | 2.00x | 1.88x |
| Singletons | 15.8% | **13.0%** |

### Trade-off
Outlier reduction maximises coverage (98.2%) at the cost of coherence (0.581 → below
traditional baseline of 0.649). The reassigned articles are borderline cases that
don't perfectly fit their nearest topic, slightly diluting story purity.

**Selected for comparison: BERTopic with outlier reduction** — the near-complete
coverage (98.2%) is the defining characteristic of Option B and the strongest
argument for a BERTopic-based approach over the traditional pipeline.


# Story Building — Method Comparison Summary

## Full Results

| Method | Coherence | Coverage | Ratio | Singletons |
|---|---|---|---|---|
| Traditional (T2=6) | 0.649 | 36.0% | 2.07x | ~75% |
| Option A (sim=0.78) | **0.779** | 38.5% | **2.51x** | ~79% |
| BERTopic (no reduction) | 0.618 | 71.8% | 2.00x | 15.8% |
| BERTopic (with reduction) | 0.581 | **98.2%** | 1.88x | **13.0%** |

---

## Best Approach by Situation

### 1. Best story quality — Option A (sim=0.78)
Highest coherence (0.779) and ratio (2.51x) at comparable coverage to traditional (38.5%).
**Use when**: story purity is the priority — e.g. timeline visualisation where
incorrect groupings are actively misleading.

### 2. Best coverage — BERTopic with outlier reduction
98.2% of articles assigned to stories, lowest singleton rate (13%).
**Use when**: completeness matters more than precision — e.g. news monitoring
where missing a story is worse than a slightly noisy one.

### 3. Best balance — BERTopic without outlier reduction
71.8% coverage with reasonable coherence (0.618) and low singletons (15.8%).
**Use when**: a middle ground is needed — good coverage without sacrificing
too much story purity.

---

## What happened to Traditional?

The traditional approach is **superseded by Option A** in every metric at
similar coverage. Replacing keyword Jaccard with BGE-M3 embeddings within
the exact same pipeline structure improves coherence by +13% and ratio by +21%.
The traditional approach remains valuable as a reproducible paper baseline
and for understanding the contribution of the embedding improvement.

---

## Key Insight

The three approaches represent three points on the **precision/recall curve**:

| ← High precision | | High recall → |
|---|---|---|
| Option A | BERTopic (no reduction) | BERTopic (with reduction) |
| coh=0.779, cov=38.5% | coh=0.618, cov=71.8% | coh=0.581, cov=98.2% |


The choice depends on the downstream task. For a news timeline system where
story clarity drives user experience, **Option A is the recommended approach**.


